Below is a compact **reference table** of the most useful **scikit‑optimize (skopt) hyperparameters** that affect the **exploration ⇄ exploitation** balance when you’re using the **Gaussian‑process** workflow (e.g., `gp_minimize`) or the lower‑level `Optimizer`. For each knob, you’ll find where to set it, common defaults, and how changing it nudges behavior.

> skopt is a *minimizer* by design; acquisition functions (EI/PI/LCB) and their knobs (`xi`, `kappa`) are the primary levers for exploration vs. exploitation. [\[github.com\]](https://github.com/github/copilot-cli), [\[linuxera.org\]](https://linuxera.org/spec-driven-development-with-spec-kit/)

***

### Key hyperparameters and their effect

| Hyperparameter                                                  | Where you set it                                                                  | Typical / default                                                                                          | Effect on Exploration vs. Exploitation                                                                                                                                        | Notes                                                                                                                                                                                                                                                                                                       |       |                        |                                                                                                             |                                                                                                                                                                                                        |
| --------------------------------------------------------------- | --------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----- | ---------------------- | ----------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| **`acq_func`** (LCB, EI, PI, gp\_hedge, …)                      | `gp_minimize(..., acq_func=...)` or `Optimizer(..., acq_func=...)`                | Defaults vary by API version (e.g., `gp_hedge` in classic `gp_minimize`, `EI` in base optimizer); see docs | **LCB** + high `kappa` ⇒ explores. **EI/PI** + low `xi` ⇒ exploits. **gp\_hedge** mixes and adapts across LCB/EI/PI during the run.                                           | skopt documents LCB/EI/PI formulas and `gp_hedge`; choose LCB for direct control via `kappa`, EI/PI for `xi` control, or `gp_hedge` to adapt automatically. [\[github.com\]](https://github.com/github/copilot-cli), [\[hub.tcno.co\]](https://hub.tcno.co/ai/cli/copilot/) |       |                        |                                                                                                             |                                                                                                                                                                                                        |
| **`xi`** (EI/PI exploration parameter)                          | `gp_minimize(..., xi=...)` or `Optimizer(..., acq_func_kwargs={'xi': ...})`       | **Default \~0.01**                                                                                         | **↑ `xi`** ⇒ encourages exploration (samples uncertain regions). **↓ `xi` → 0** ⇒ greedy exploitation.                                                                        | Applies to **EI**/**PI**. It offsets the improvement threshold in the acquisition, trading off exploration vs. exploitation. [\[datacamp.com\]](https://www.datacamp.com/tutorial/github-copilot-cli-tutorial), [\[github.com\]](https://github.com/github/copilot-cli)      |       |                        |                                                                                                             |                                                                                                                                                                                                        |
| **`kappa`** (LCB exploration parameter)                         | `gp_minimize(..., kappa=...)` or `Optimizer(..., acq_func_kwargs={'kappa': ...})` | **Default \~1.96**                                                                                         | **↑ `kappa`** ⇒ more exploration (larger uncertainty bonus). **↓ `kappa`** ⇒ more exploitation (trusts the mean).                                                             | Used only with **LCB**; LCB(x)=μ−κ·σ—so κ directly scales how much variance you seek. [\[datacamp.com\]](https://www.datacamp.com/tutorial/github-copilot-cli-tutorial), [\[github.com\]](https://github.com/github/copilot-cli)                                             |       |                        |                                                                                                             |                                                                                                                                                                                                        |
| **`n_initial_points`**                                          | `gp_minimize(..., n_initial_points=...)` / `Optimizer(..., n_initial_points=...)` | **Default 10**                                                                                             | **↑** ⇒ broader *model‑free* exploration up front; **↓** ⇒ begins model‑guided exploitation sooner.                                                                           | Initial points are sampled before BO uses the surrogate; they shape early coverage of the space. [\[linuxera.org\]](https://linuxera.org/spec-driven-development-with-spec-kit/), [\[hub.tcno.co\]](https://hub.tcno.co/ai/cli/copilot/)                                      |       |                        |                                                                                                             |                                                                                                                                                                                                        |
| **`initial_point_generator`**                                   | \`... initial\_point\_generator="random"                                          | "sobol"                                                                                                    | "halton"                                                                                                                                                                      | "lhs"                                                                                                                                                                                                                                                                                                       | ...\` | **Default `"random"`** | **Sobol/Halton/LHS** ⇒ more *space‑filling* exploration; `"random"` tends to be less uniformly exploratory. | Low‑discrepancy sequences (Sobol/Halton) improve early coverage; helpful if you want stronger initial exploration. [\[hub.tcno.co\]](https://hub.tcno.co/ai/cli/copilot/) |
| **`acq_optimizer`**, **`n_points`**, **`n_restarts_optimizer`** | `gp_minimize(..., acq_optimizer=..., n_points=..., n_restarts_optimizer=...)`     | `acq_optimizer="auto"`, `n_points=10000`, `n_restarts_optimizer=5` (classic defaults)                      | **↑ `n_points`/`n_restarts_optimizer`** lets the acquisition’s internal search roam wider ⇒ tends to propose more exploratory points when warranted.                          | These control how thoroughly the acquisition is maximized (or sampled) each iteration; broader search = more chance to pick uncertain regions. [\[linuxera.org\]](https://linuxera.org/spec-driven-development-with-spec-kit/)                                                  |       |                        |                                                                                                             |                                                                                                                                                                                                        |
| **`base_estimator`** (surrogate model)                          | `gp_minimize(..., base_estimator=...)` / `Optimizer(..., base_estimator=...)`     | **Default GP**                                                                                             | **GP** provides calibrated μ,σ ⇒ acquisition balances exploration/exploitation as designed. Using RF/ET/GBRT changes uncertainty behavior (often more heuristic).             | For EI/LCB to work well, the surrogate must expose **mean + std**; GP is the standard for this in skopt. [\[hub.tcno.co\]](https://hub.tcno.co/ai/cli/copilot/)                                                                                                                |       |                        |                                                                                                             |                                                                                                                                                                                                        |
| **`noise`** (GP)                                                | `gp_minimize(..., noise=...)`                                                     | e.g., `"gaussian"` (GP with noise)                                                                         | **More noise** can keep predictive σ from collapsing too quickly ⇒ allows sustained exploration; **too little noise** can lead to over‑confidence and premature exploitation. | GP + noise is the default in `gp_minimize`; tune only if you understand your objective’s noise profile. [\[linuxera.org\]](https://linuxera.org/spec-driven-development-with-spec-kit/)                                                                                         |       |                        |                                                                                                             |                                                                                                                                                                                                        |
| **`gp_hedge` strategy**                                         | `acq_func="gp_hedge"`                                                             | —                                                                                                          | Lets skopt **adaptively** switch among EI/PI/LCB per step ⇒ *dynamic* exploration/exploitation.                                                                               | Good default when you don’t want to hand‑tune `xi`/`kappa`—the strategy maintains running gains and picks per‑step. [\[hub.tcno.co\]](https://hub.tcno.co/ai/cli/copilot/)                                                                                                     |       |                        |                                                                                                             |                                                                                                                                                                                                        |
| **`n_calls`** (overall budget)                                  | `gp_minimize(..., n_calls=...)`                                                   | **Default 100**                                                                                            | **Higher budget** permits more exploration *and* later exploitation; **very small budgets** favor greedy behavior (less time to explore).                                     | Not an acquisition hyperparameter, but it strongly conditions the exploration window available. [\[linuxera.org\]](https://linuxera.org/spec-driven-development-with-spec-kit/)                                                                                                 |       |                        |                                                                                                             |                                                                                                                                                                                                        |

***

## Quick recipes

*   **Exploration‑forward (EI):**  
    `acq_func="EI", xi=0.05–0.10, n_initial_points=20, initial_point_generator="sobol", n_points=20000, n_restarts_optimizer=20`  
    (Higher `xi` + more/better initial coverage + broader acquisition search.) [\[github.com\]](https://github.com/github/copilot-cli), [\[hub.tcno.co\]](https://hub.tcno.co/ai/cli/copilot/), [\[linuxera.org\]](https://linuxera.org/spec-driven-development-with-spec-kit/)

*   **Exploration‑forward (LCB):**  
    `acq_func="LCB", kappa=2.5–5.0, n_initial_points=20, initial_point_generator="halton"`  
    (Large `kappa` directly rewards uncertainty; space‑filling starts help.) [\[github.com\]](https://github.com/github/copilot-cli)

*   **Exploit‑first (EI):**  
    `acq_func="EI", xi=0.0–0.005, n_initial_points=5, initial_point_generator="random"`  
    (Greedy improvement; minimal warm‑up to start exploiting early.) [\[github.com\]](https://github.com/github/copilot-cli), [\[linuxera.org\]](https://linuxera.org/spec-driven-development-with-spec-kit/)

*   **Exploit‑first (LCB):**  
    `acq_func="LCB", kappa=0.5–1.0, n_initial_points=5`  
    (Trust the mean; small κ reduces the uncertainty bonus.) [\[github.com\]](https://github.com/github/copilot-cli)

***

## Pointers to the docs & source

*   **Acquisitions & knobs** (`EI`/`PI`/`LCB`, `xi`, `kappa`): skopt user guide and API. [\[github.com\]](https://github.com/github/copilot-cli)
*   **`gp_minimize` arguments** (including `xi`, `kappa`, `n_points`, `n_restarts_optimizer`, `noise`): API reference. [\[linuxera.org\]](https://linuxera.org/spec-driven-development-with-spec-kit/)
*   **`Optimizer` arguments** (initial designs, `gp_hedge`, etc.): API reference. [\[hub.tcno.co\]](https://hub.tcno.co/ai/cli/copilot/), [\[codestandup.com\]](https://codestandup.com/posts/2025/github-spec-kit-tutorial-constitution-command/)
*   **Defaults in code** (`xi≈0.01`, `kappa≈1.96`, acquisition internals): acquisition module. [\[datacamp.com\]](https://www.datacamp.com/tutorial/github-copilot-cli-tutorial)
